In [17]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from src.tmdb.client import TMDBClient
from src.tmdb.cache import TMDBCache

ImportError: cannot import name 'TMDBCache' from partially initialized module 'src.tmdb.cache' (most likely due to a circular import) (/Users/smac4/Desktop/Unified Recommender/OmniRec/src/tmdb/cache.py)

In [4]:
client = TMDBClient()
movie = client.get_movie_details(603)
print(movie["title"])

APIRequestError: Network error : HTTPSConnectionPool(host='api.themoviedb.org', port=443): Max retries exceeded with url: /3/movie/603?api_key=28048bc8e64a2c1f64965420f9ad6ce3 (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.themoviedb.org', port=443) at 0x11da16ba0>, 'Connection to api.themoviedb.org timed out. (connect timeout=30)'))

In [3]:
import requests

response = requests.get("https://www.google.com", timeout=10)

print(response.status_code)

200


In [ ]:
import requests

url = "https://api.themoviedb.org/3/movie/603"

params = {
    "api_key": "28048bc8e64a2c1f64965420f9ad6ce3"
}

response = requests.get(url, params=params, timeout=20)

print(response.status_code)
print(response.text[:200])

ConnectTimeout: HTTPSConnectionPool(host='api.themoviedb.org', port=443): Max retries exceeded with url: /3/movie/603?api_key=28048bc8e64a2c1f64965420f9ad6ce3 (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.themoviedb.org', port=443) at 0x11203f0b0>, 'Connection to api.themoviedb.org timed out. (connect timeout=20)'))

In [5]:
class FakeClient:
    def get_movie_details(self, movie_id):
        return {"id": movie_id, "title": "Fake Movie"}

    def get_movie_credits(self, movie_id):
        return {"cast": []}

    def get_movie_keywords(self, movie_id):
        return {"keywords": []}

In [6]:
call = FakeClient()

In [8]:
call.get_movie_details(101)

{'id': 101, 'title': 'Fake Movie'}

In [9]:
call.get_movie_credits(101)

{'cast': []}

In [10]:
call.get_movie_keywords(101)

{'keywords': []}

In [5]:
from src.tmdb.enricher import TMDBEnricher

In [10]:
details = {
    "genres": [
        {"id":28,"name":"Action"},
        {"id":878,"name":"Science Fiction"},
        {"id":12,"name":"Adventure"}
    ]
}

In [7]:
genres = [
    "Action",
    "Science Fiction",
    "Adventure"
]

In [8]:
enricher = TMDBEnricher()

In [12]:
enricher._extract_genres(details)

'Action Science_Fiction Adventure'

In [14]:
keywords = {
    
    "id": 603,
    "keywords": [
        {
            "id": 825,
            "name": "artificial intelligence"
        },
        {
            "id": 156095,
            "name": "hacker"
        },
        {
            "id": 9715,
            "name": "virtual reality"
        }
    ]

}

In [15]:
enricher._extract_keywords(keywords)

'artificial_intelligence hacker virtual_reality'

In [19]:
credits = {
    "id": 603,

    "cast": [
        {
            "name": "Keanu Reeves",
            "character": "Neo",
            "order": 0
        },
        ...
    ],

    "crew": [
        {
            "name": "Lana Wachowski",
            "job": "Director"
        },
        {
            "name": "Lilly Wachowski",
            "job": "Director"
        },
        {
            "name": "Bill Pope",
            "job": "Director of Photography"
        },
        {
            "name": "Don Davis",
            "job": "Original Music Composer"
        }
    ]
}

In [20]:
enricher._extract_director(credits)

'Lana_Wachowski Lilly_Wachowski'

In [23]:

import pandas as pd

from src.tmdb.client import TMDBClient
from src.tmdb.cache import TMDBCache
from src.tmdb.fetcher import TMDBFetcher
from src.tmdb.enricher import TMDBEnricher
from src.pipelines.enrichment_pipeline import EnrichmentPipeline

In [ ]:
# import os
# from pathlib import Path

# # 1. Print current working directory
# print("Current Working Dir:", os.getcwd())

# # 2. Check if the directory and file exist
# data_path = Path("data/processed/movie_features.csv")
# print("Target file absolute path:", data_path.resolve())
# print("Does file exist?:", data_path.exists())

Current Working Dir: /Users/smac4/Desktop/Unified Recommender/OmniRec/notebooks
Target file absolute path: /Users/smac4/Desktop/Unified Recommender/OmniRec/notebooks/data/processed/movie_features.csv
Does file exist?: False


In [27]:
from pathlib import Path
import pandas as pd

# Finds the root directory assuming notebook is in a subdirectory (like 'notebooks/')
project_root = Path.cwd().parent  # or Path.cwd() depending on where your notebook is
file_path = project_root / "data" / "processed" / "movie_features.csv"

movie_features = pd.read_csv(file_path)
print(f"Movies loaded: {len(movie_features)}")

Movies loaded: 9742


In [28]:
client = TMDBClient()
cache = TMDBCache()

fetcher = TMDBFetcher(
    client=client,
    cache=cache,
)

enricher = TMDBEnricher()

pipeline = EnrichmentPipeline(
    fetcher=fetcher,
    enricher=enricher,
)

In [31]:
sample_movies = movie_features.head(10)

print(f"Testing on {len(sample_movies)} movies...\n")

enriched_df = pipeline.enrich_movies(sample_movies)

Testing on 10 movies...



In [32]:
print("\nEnrichment completed successfully!\n")

print("Shape:")
print(enriched_df.shape)

print("\nColumns:")
print(enriched_df.columns.tolist())

print("\nPreview:")
display(enriched_df.head())


Enrichment completed successfully!

Shape:
(10, 19)

Columns:
['movieId', 'title', 'genres', 'average_rating', 'rating_count', 'tags', 'imdbId', 'tmdbId', 'metadata', 'overview', 'keywords', 'director', 'cast', 'runtime', 'release_year', 'vote_average', 'vote_count', 'popularity', 'original_language']

Preview:


,movieId,title,genres,average_rating,rating_count,tags,imdbId,tmdbId,metadata,overview,keywords,director,cast,runtime,release_year,vote_average,vote_count,popularity,original_language
0,1,Toy Story,Family Comedy Animation Adventure,3.92,215.0,pixar pixar fun,114709,862,adventure animation children comedy fantasy pi...,"Led by Woody, Andy's toys live happily in his ...",rescue friendship mission jealousy villain bul...,John_Lasseter,Tom_Hanks Tim_Allen Don_Rickles Jim_Varney Wal...,81,1995,7.983,20176,30.9498,en
1,2,Jumanji,Adventure Fantasy Family,3.43,110.0,fantasy magic board game Robin Williams game,113497,8844,adventure children fantasy magic board game ro...,When siblings Judy and Peter discover an encha...,based_on_novel_or_book giant_insect board_game...,Joe_Johnston,Robin_Williams Kirsten_Dunst Bradley_Pierce Bo...,104,1995,7.249,11396,2.4359,en
2,3,Grumpier Old Men,Romance Comedy,3.26,52.0,moldy old,113228,15602,comedy romance moldy old,A family wedding reignites the ancient feud be...,fishing sequel old_man best_friend wedding ita...,Howard_Deutch,Walter_Matthau Jack_Lemmon Ann-Margret Sophia_...,101,1995,6.479,432,1.6558,en
3,4,Waiting to Exhale,Comedy Drama Romance,2.36,7.0,NaN,114885,31357,comedy drama romance,"Cheated on, mistreated and stepped on, the wom...",based_on_novel_or_book single_mother divorce a...,Forest_Whitaker,Whitney_Houston Angela_Bassett Loretta_Devine ...,127,1995,6.261,207,1.8246,en
4,5,Father of the Bride Part II,Comedy Family,3.07,49.0,pregnancy remake,113041,11862,comedy pregnancy remake,Just when George Banks has recovered from his ...,daughter baby parent_child_relationship midlif...,Charles_Shyer,Steve_Martin Diane_Keaton Martin_Short Kimberl...,106,1995,6.271,819,2.1537,en


In [34]:
output_path = "data/processed/enriched_movie_features_test.csv"

enriched_df.to_csv(
    output_path,
    index=False,
)

print(f"\nSaved to: {output_path}")

OSError: Cannot save file into a non-existent directory: 'data/processed'

In [33]:
enriched_df.iloc[0]

movieId                                                              1
title                                                        Toy Story
genres                               Family Comedy Animation Adventure
average_rating                                                    3.92
rating_count                                                     215.0
tags                                                   pixar pixar fun
imdbId                                                          114709
tmdbId                                                             862
metadata             adventure animation children comedy fantasy pi...
overview             Led by Woody, Andy's toys live happily in his ...
keywords             rescue friendship mission jealousy villain bul...
director                                                 John_Lasseter
cast                 Tom_Hanks Tim_Allen Don_Rickles Jim_Varney Wal...
runtime                                                             81
releas